# Encoder stream ablation: bisakah satu encoder dibuang?

Buka dari root **SongMap**. Notebook memakai Python `.venv-songformer311` di sibling Section-Annotations secara eksplisit; kernel notebook boleh berbeda.

**Tujuan:** ukuran lebih kecil dan analisis lebih cepat dengan akurasi worship tetap terjaga. Encoder MusicFM + MuQ adalah ~352 MiB dari aset model, dan keduanya juga mendominasi waktu analisis. Notebook ini mengukur apakah head SongFormer tetap akurat bila hanya memakai sebagian stream:

| Varian | Stream | Bank encoder | Pass encoder |
|---|---|---:|---:|
| `all4` (kontrol) | MusicFM30 + MuQ30 + MusicFM120 + MuQ120 | 100% | 100% |
| `musicfm` | MusicFM30 + MusicFM120 | ~50% | 50% |
| `muq` | MuQ30 + MuQ120 | ~50% | 50% |
| `short30` | MusicFM30 + MuQ30 | 100% | 50% |

Hanya cache fitur layer-10 yang sudah ada yang dibaca. Tidak menjalankan encoder, tidak decoding audio, label manual/boundary/split tidak berubah, dan **split test tidak dibaca**. Semua varian mulai dari `worship-v3/best.pt` dengan resep training yang sama; kolom proyeksi input untuk stream yang dibuang dihapus. Kontrol `all4` dilatih dengan resep yang sama agar perbandingannya adil.

Aturan keputusan ditetapkan **sebelum** hasil ada: varian menjadi *kandidat* bila akurasi section dan macro F1 validation masing-masing turun maksimal 0,01 dibanding kontrol `all4`. Interval bootstrap per lagu dilaporkan sebagai informasi ketidakpastian (32 lagu validation relatif kecil).

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, json
assert Path('scripts/encoder_stream_ablation.py').is_file(), 'Buka notebook dari root SongMap'
SONGSTRUCT_ROOT=Path('../Section-Annotations').resolve()
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
CHECKPOINT=SONGSTRUCT_ROOT/'outputs.local/finetuning/runs/worship-v3/best.pt'
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'
assert CHECKPOINT.is_file(), f'Checkpoint tidak ditemukan: {CHECKPOINT}'
DEVICE='mps'
TRAINABLE='projection_head'   # proyeksi input + LayerNorm + function head; 'label_backbone' untuk tahap lanjut
OUTPUT=Path('outputs.local')/('stream-ablation-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
subprocess.run([str(RUN_PYTHON),'-c',"import sys,importlib.util; missing=[p for p in ('numpy','torch','scipy','omegaconf','safetensors','ema_pytorch') if importlib.util.find_spec(p) is None]; print(sys.executable); sys.exit('Dependency missing: '+', '.join(missing)) if missing else print('Dependency check OK')"],check=True)
print('Checkpoint:',CHECKPOINT)
print('Output:',OUTPUT.resolve())

In [ ]:
def run(*arguments,name):
    OUTPUT.mkdir(parents=True,exist_ok=True)
    log=OUTPUT/(name+'.log')
    command=[str(RUN_PYTHON),'-u','scripts/encoder_stream_ablation.py',*arguments]
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:
        raise RuntimeError(f'Gagal ({status}); lihat {log.resolve()}')

def train(variant,smoke=False):
    run('train','--variant',variant,'--init-checkpoint',str(CHECKPOINT),'--output-dir',str(OUTPUT),
        '--songstruct-root',str(SONGSTRUCT_ROOT),'--device',DEVICE,'--trainable',TRAINABLE,
        *(['--smoke-test'] if smoke else []),name=variant+('-smoke' if smoke else ''))

# Satu langkah backward per varian; tidak menyimpan apa pun.
for variant in ('all4','musicfm','muq','short30'):
    train(variant,smoke=True)

Kontrol dulu. Epoch 0 dari `all4` adalah `worship-v3` tanpa perubahan; `summary.json` mencatat `init_reproduced`. Bila `false`, target validation sudah berubah sejak `worship-v3` dilatih — tetap aman karena semua varian dibandingkan dengan kontrol `all4` yang dilatih ulang di sini.

Periksa log `all4` sebelum menjalankan varian: loss training harus cenderung turun dan validation tidak boleh anjlok tiap epoch. Default resep: window 30 detik, akumulasi 56 window (≈ audio per langkah sama dengan 4 chunk 420 detik `worship-v3`), learning rate 1e-5. Learning rate 1e-4 dengan akumulasi 4 sebelumnya membuat loss training naik lagi sejak epoch 2 dan F1 turun 0,7238 → 0,5791. Bila `all4` tidak membaik, best tetap epoch 0 (= `worship-v3`) dan perbandingan tetap sah; resep yang stabil tetap penting agar varian bisa pulih dari kolom yang dibuang.


In [ ]:
train('all4')

In [ ]:
for variant in ('musicfm','muq','short30'):
    train(variant)

Opsional, hanya bila `musicfm` atau `muq` lolos dan Anda ingin mengecilkan lagi: varian satu stream (`musicfm30`, `muq30`) mempertahankan satu bank encoder dan hanya 25% pass encoder.

In [ ]:
# for variant in ('musicfm30','muq30'):
#     train(variant)

Perbandingan berpasangan pada section validation yang identik. Isi `ENCODER_BYTES` dengan ukuran compiled `.mlmodelc` setiap bank canonical (byte) agar kolom ukuran memakai ukuran sebenarnya, bukan asumsi 50/50.

In [ ]:
ENCODER_BYTES=None   # contoh: 'musicfm=184000000,muq=185000000'
run('compare','--output-dir',str(OUTPUT),*(['--encoder-bytes',ENCODER_BYTES] if ENCODER_BYTES else []),name='compare')
report=json.loads((OUTPUT/'comparison.json').read_text())
for variant,entry in report['variants'].items():
    c=entry.get('comparison')
    if c:
        print(variant,'bootstrap 95% dAcc',[round(v,4) for v in c['bootstrap_interval']['section_accuracy']],
              'dF1',[round(v,4) for v in c['bootstrap_interval']['macro_f1_present']],
              'berubah',len(c['changed_sections']),'section')
print('Bagikan:',(OUTPUT/'comparison.json').resolve())

**Cara membaca.** `candidate` hanya berarti lolos aturan non-inferiority di validation terhadap kontrol yang dilatih sama. Ini bukan hasil test, bukan hasil pipeline dengan model konteks, dan bukan hasil Core ML/iPhone.

Bila ada varian `candidate`, langkah berikutnya (dikerjakan terpisah):
1. Latih ulang model konteks section dengan fitur pooled dari stream yang dipertahankan, lalu bandingkan dengan pipeline 79,16% / 0,7672 sekarang.
2. Evaluasi sekali pada split test untuk varian terpilih saja.
3. Ekspor head dengan input 2048, palette6, dan siapkan probe iPhone tanpa encoder yang dibuang: ukuran dan waktu diukur, bukan diasumsikan.

Bila semua varian `rejected`, kedua encoder memang dibutuhkan; jalur pengecilan berikutnya adalah palettization encoder di bawah 6 bit dengan evaluasi validation penuh.

## Percobaan kedua: MuQ saja, window training 120 detik

Hasil pertama: tidak ada varian yang lolos. `muq` paling dekat (akurasi −0,0067, macro F1 −0,0265) dan terbaik di epoch 0, sedangkan training dengan window 30 detik tidak memperbaiki `all4` maupun `muq`. Window 120 detik lebih dekat ke chunk inference 420 detik. Ini percobaan **kedua dan terakhir** untuk jalur satu-encoder, dengan aturan keputusan yang sama (maks. turun 0,01 untuk akurasi dan macro F1 terhadap `all4` yang dilatih dengan resep yang sama). Bila gagal, jalur satu-encoder ditutup.

Resep: window 120 detik, akumulasi otomatis 14 window (audio per langkah sama), learning rate 1e-5. Folder output baru; hasil pertama tidak ditimpa. Bila MPS kehabisan memori, ganti `WINDOW_SECONDS2` ke 60 lalu jalankan ulang sel ini dari awal.

In [ ]:
WINDOW_SECONDS2=120
OUTPUT2=Path('outputs.local')/('stream-ablation-w120-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
def train2(variant,smoke=False):
    global OUTPUT
    previous,OUTPUT=OUTPUT,OUTPUT2
    try:
        run('train','--variant',variant,'--init-checkpoint',str(CHECKPOINT),'--output-dir',str(OUTPUT2),
            '--songstruct-root',str(SONGSTRUCT_ROOT),'--device',DEVICE,'--trainable',TRAINABLE,
            '--window-seconds',str(WINDOW_SECONDS2),*(['--smoke-test'] if smoke else []),
            name=variant+('-smoke' if smoke else ''))
    finally:
        OUTPUT=previous
for variant in ('all4','muq'):
    train2(variant,smoke=True)
print('Output:',OUTPUT2.resolve())

In [ ]:
for variant in ('all4','muq'):
    train2(variant)
OUTPUT,previous=OUTPUT2,OUTPUT
try:
    run('compare','--output-dir',str(OUTPUT2),name='compare')
finally:
    OUTPUT=previous
c=json.loads((OUTPUT2/'comparison.json').read_text())['variants']['muq']['comparison']
print('muq dAcc',round(c['delta_section_accuracy'],4),'dF1',round(c['delta_macro_f1_present'],4),'->',c['verdict'])
print('Bagikan:',(OUTPUT2/'comparison.json').resolve())